# Week 4 · Tutorial 2 — Inconsistencies, new features, rescaling and encoding categories

**Covers videos 4.4 – 4.7:** data inconsistencies · feature engineering · univariate rescaling · categorical encoding.

**Datasets**
* `power_plants.csv` — a register of 900 power plants. **One row is one plant**: `fuel` (typed by hand!), `capacity_mw`, `annual_gwh` (energy produced last year), `commissioned`, `municipality`, `region`.
* `district_heat_daily.csv` — daily outdoor temperature and heat delivered by a district-heating network (MWh), 2021–2023. **One row is one day.**
* `day_ahead_prices.csv` — hourly spot prices (€/MWh), 2022–2023.

Everything that is *fitted* (scalers, encodings) is fitted on the **training** rows only: here the first 70 % of the rows (plants: in file order; days and hours: in time).

### How to work with this notebook (no programming experience needed)

* A notebook is a list of **cells**. Click a code cell and press **Shift + Enter** to run it; the result appears below it. Run the cells **from top to bottom**.
* Cells that start with `import` or read a file (`pd.read_csv(...)`) just load tools and data. You do not need to understand every line.
* In every **✏️ Your turn** block you make one small, exactly described change (a number, a word, or a line to paste in), then run the cell again.
  The box **What to change** shows the exact text to type. Lines marked `# TODO` are the ones to edit.
* Write down the numbers that appear — the quiz asks for them. If you break something, re-run the cells above it, or use *Kernel → Restart & Run All*.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 120)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})

## 1. Inconsistencies: errors that look like valid data

In [ ]:
plants = pd.read_csv("data/power_plants.csv")
print(plants.shape)
plants.head()

### One column, many spellings

In [ ]:
print("distinct raw spellings in fuel:", plants["fuel"].nunique())
print(plants["fuel"].value_counts().head(12))

### ✏️ Your turn 1

Clean the `fuel` column in two steps:

1. **Strip and lowercase**: `plants["fuel"].str.strip().str.lower()`
2. **Map synonyms** to one name each: `"wind power"` → `"wind"`, `"hydropower"` → `"hydro"`, `"solar pv"` → `"solar"`, `"natural gas"` → `"gas"` (use `.replace(synonyms)`).

The cell reports the number of distinct values after each step, and the number of wind plants.

**What to change** — edit the cell exactly like this, then run it:

Replace
```python
step1 = plants["fuel"]
step2 = step1
```
by
```python
step1 = plants["fuel"].str.strip().str.lower()
step2 = step1.replace(synonyms)
```

In [ ]:
synonyms = {"wind power": "wind", "hydropower": "hydro", "solar pv": "solar", "natural gas": "gas"}

step1 = plants["fuel"]               # TODO: strip + lowercase
step2 = step1                        # TODO: map the synonyms

plants["fuel_clean"] = step2
print("distinct spellings: raw", plants["fuel"].nunique(), "| after step 1:", step1.nunique(), "| after step 2:", step2.nunique())
print("wind plants:", int((plants["fuel_clean"] == "wind").sum()))
print(plants["fuel_clean"].value_counts())

### Columns lie — relationships are harder to fake

A plant of capacity *C* MW can produce at most *C* × 8.76 GWh per year. The **capacity factor** `annual_gwh / (capacity_mw · 8.76)`
is therefore between 0 and 1 — real plants lie between roughly 5 % and 95 %. If the capacity is entered in kW instead of MW, the factor collapses to nearly 0.

### ✏️ Your turn 2

Find and repair unit errors.

1. Change `min_plausible` from 0.0 to **0.01**: capacity factors below 1 % are physically implausible for these plants.
2. Fill in the `TODO`: for the flagged plants the capacity was entered in kW instead of MW; divide it by 1000 (`plants.loc[bad, "capacity_mw"] / 1000`).
3. The cell recomputes the capacity factor and reports the numbers before and after the repair.

**What to change** — edit the cell exactly like this, then run it:

1. Change `min_plausible = 0.0` into `min_plausible = 0.01`

2. Change `# TODO: plants.loc[bad, "capacity_mw"] = ... / 1000` into `plants.loc[bad, "capacity_mw"] = plants.loc[bad, "capacity_mw"] / 1000`

In [ ]:
plants["cf"] = plants["annual_gwh"] / (plants["capacity_mw"] * 8.76)

min_plausible = 0.0          # TODO: 0.01
bad = plants["cf"] < min_plausible

total_before = plants["capacity_mw"].sum()
# TODO: plants.loc[bad, "capacity_mw"] = ... / 1000
plants["cf"] = plants["annual_gwh"] / (plants["capacity_mw"] * 8.76)

print("implausible capacity factors:", int(bad.sum()))
print("total capacity before / after repair (GW):", round(total_before / 1000, 1), round(plants["capacity_mw"].sum() / 1000, 1))
print("capacity factor range after repair:", round(plants["cf"].min(), 3), round(plants["cf"].max(), 3))
print("highest mean capacity factor:", plants.groupby("fuel_clean")["cf"].mean().idxmax(),
      round(plants.groupby("fuel_clean")["cf"].mean().max(), 3))

## 2. Feature engineering: changing the shape of the question

The district-heating load does not depend linearly on temperature: above a **base temperature** nobody heats, and the load is
flat; below it, the load grows with every degree. A straight line is the wrong summary. The engineered feature
**heating degrees** = `max(0, T0 − temperature)` makes the relation linear. *You add no information — you change the shape.*

In [ ]:
dh = pd.read_csv("data/district_heat_daily.csv", parse_dates=["date"]).sort_values("date")
dh_train = dh.iloc[:int(0.7 * len(dh))]
print("training days:", len(dh_train))

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.scatter(dh_train["temp_c"], dh_train["heat_load_mwh"], s=5, alpha=0.5)
ax.set_xlabel("outdoor temperature (°C)"); ax.set_ylabel("heat load (MWh/day)")
plt.show()
print("correlation load vs temperature:", round(dh_train["heat_load_mwh"].corr(dh_train["temp_c"]), 3))

### ✏️ Your turn 3

Find the best base temperature.

Fill in the `TODO`: for every candidate `T0` in `range(8, 23)` compute the feature `np.maximum(0, T0 - temp_c)` on the training days and
its correlation with `heat_load_mwh`; store them in `corr_by_T0`. Then the cell reports the best `T0`, the best correlation (3 decimals) and
how much of the variance of the load the feature explains (r², 3 decimals) compared with the raw temperature.

**What to change** — edit the cell exactly like this, then run it:

Replace
```python
    pass
```
by
```python
    feature = np.maximum(0, T0 - dh_train["temp_c"])
    corr_by_T0[T0] = dh_train["heat_load_mwh"].corr(feature)
```

In [ ]:
corr_by_T0 = {}
for T0 in range(8, 23):
    pass      # TODO: corr_by_T0[T0] = correlation of np.maximum(0, T0 - temp) with the heat load

if corr_by_T0:
    best_T0 = max(corr_by_T0, key=corr_by_T0.get)
    r_best = corr_by_T0[best_T0]
    r_raw = dh_train["heat_load_mwh"].corr(dh_train["temp_c"])
    print("best base temperature:", best_T0)
    print("correlation with that feature:", round(r_best, 3))
    print("correlation with T0 = 15:", round(corr_by_T0[15], 3))
    print("r2 raw temperature:", round(r_raw ** 2, 3), "| r2 heating degrees:", round(r_best ** 2, 3))

## 3. Univariate rescaling

Spot prices are right-skewed, with negative values and rare spikes. Rescaling changes the numbers, not the information.

| transformation | when |
|---|---|
| log | skewed and **positive** |
| Yeo–Johnson | skewed, any sign |
| robust scaling `(x − median) / IQR` | outliers present |
| clip | a real threshold |

In [ ]:
from sklearn.preprocessing import PowerTransformer

pr = pd.read_csv("data/day_ahead_prices.csv", parse_dates=["timestamp"]).sort_values("timestamp")
p_train = pr.iloc[:int(0.7 * len(pr))]["price_eur_mwh"]
p_rest = pr.iloc[int(0.7 * len(pr)):]["price_eur_mwh"]
print("training hours:", len(p_train), "| negative prices:", int((p_train < 0).sum()), "| max:", p_train.max())
print("skewness of the raw price:", round(p_train.skew(), 2))

### ✏️ Your turn 4

Fill in the three `TODO`s (all fitted on `p_train`):
- `skew_log`: skewness of `np.log(p_train[p_train > 0])` (only the positive prices can be logged) (2 decimals),
- `skew_yj`: skewness of the Yeo–Johnson transformed price (`PowerTransformer(method="yeo-johnson")` fitted on `p_train`) (2 decimals),
- `robust`: the robustly scaled price `(p − median) / IQR` with median and IQR of `p_train`, and `n_extreme`: the number of training hours with `|robust| > 3`.

**What to change** — edit the cell exactly like this, then run it:

1. Add these lines after `pt = PowerTransformer(method="yeo-johnson")`:
```python
yj_train = pt.fit_transform(p_train.to_frame())[:, 0]
```

2. Replace
```python
skew_log = np.nan
skew_yj = np.nan
n_extreme = np.nan
```
by
```python
skew_log = np.log(p_train[p_train > 0]).skew()
skew_yj = pd.Series(yj_train).skew()
iqr = p_train.quantile(0.75) - p_train.quantile(0.25)
robust = (p_train - p_train.median()) / iqr
n_extreme = int((robust.abs() > 3).sum())
```

In [ ]:
pt = PowerTransformer(method="yeo-johnson")

skew_log = np.nan        # TODO
skew_yj = np.nan         # TODO
n_extreme = np.nan       # TODO: hours with |(p - median) / IQR| > 3

print("skew raw / log / yeo-johnson:", round(p_train.skew(), 2), round(skew_log, 2), round(skew_yj, 2))
print("hours with |robust| > 3:", n_extreme)

### ✏️ Your turn 5

The fitted transformation is applied to the *rest* of the data — with the parameters from the training data. Fill in the `TODO`: apply `pt` (fitted
above) to `p_rest` with `pt.transform(...)`, and report the mean and the standard deviation of the transformed rest (2 decimals).
Are they exactly 0 and 1?

**What to change** — edit the cell exactly like this, then run it:

1. Change `yj_rest = np.zeros(len(p_rest))` into `yj_rest = pt.transform(p_rest.to_frame())[:, 0]`

2. Replace
```python
print("mean / std of the transformed training prices:", round(float(np.mean(pt.transform(p_train.to_frame()))), 2) if hasattr(pt, "lambdas_") else None,
      round(float(np.std(pt.transform(p_train.to_frame()))), 2) if hasattr(pt, "lambdas_") else None)
```
by
```python
print("mean / std of the transformed training prices:", round(float(np.mean(pt.transform(p_train.to_frame()))), 2),
      round(float(np.std(pt.transform(p_train.to_frame()))), 2))
```

In [ ]:
yj_rest = np.zeros(len(p_rest))        # TODO: pt.transform(p_rest.to_frame())[:, 0]

print("mean / std of the transformed training prices:", round(float(np.mean(pt.transform(p_train.to_frame()))), 2) if hasattr(pt, "lambdas_") else None,
      round(float(np.std(pt.transform(p_train.to_frame()))), 2) if hasattr(pt, "lambdas_") else None)
print("mean / std of the transformed rest          :", round(float(yj_rest.mean()), 2), round(float(yj_rest.std()), 2))

## 4. Categorical encoding: names into numbers, without inventing an order

The `region` column has 6 levels, but `municipality` has dozens — one-hot encoding turns every level into a column, which becomes unwieldy.
The first remedy is to **group rare levels** into `"other"`.

In [ ]:
print("levels in region:", plants["region"].nunique(), "| in municipality:", plants["municipality"].nunique())
print("one-hot columns for region:", pd.get_dummies(plants["region"]).shape[1])
print(plants["municipality"].value_counts().head(8))

### ✏️ Your turn 6

Group the rare municipalities.

Change `min_count` from 1 to **10**: municipalities with fewer than 10 plants are renamed `"other"`. Then the cell reports the number of levels and the
number of one-hot columns, and the share of plants that ended up in `"other"` (3 decimals). Also run it with `min_count = 25`.

In [ ]:
min_count = 1       # TODO: 10, then 25

counts = plants["municipality"].map(plants["municipality"].value_counts())
plants["muni_grouped"] = plants["municipality"].where(counts >= min_count, "other")

print("min_count =", min_count)
print("levels after grouping:", plants["muni_grouped"].nunique())
print("one-hot columns:", pd.get_dummies(plants["muni_grouped"]).shape[1])
print("share of plants in 'other':", round((plants["muni_grouped"] == "other").mean(), 3))

### Target encoding: compact and dangerous

Replace every municipality by the **mean target value of the training plants in that municipality**. Here the target is the capacity factor `cf`.
The mean must come from the **training rows only**; municipalities not seen in training get the overall training mean.

### ✏️ Your turn 7

Fill in the `TODO`s (the first 70 % of the plants, in file order, form the training set):
- `means`: mean `cf` per `municipality` in `train_p`;
- `test_encoded`: for each test plant the value from `means`, or `global_mean` if the municipality is missing from `means` (`.map(means).fillna(global_mean)`);
- `n_unseen`: the number of test plants whose municipality does not occur in the training set.

**What to change** — edit the cell exactly like this, then run it:

Replace
```python
means = pd.Series(dtype=float)
test_encoded = pd.Series(global_mean, index=test_p.index)
n_unseen = np.nan
```
by
```python
means = train_p.groupby("municipality")["cf"].mean()
test_encoded = test_p["municipality"].map(means).fillna(global_mean)
n_unseen = int((~test_p["municipality"].isin(means.index)).sum())
```

In [ ]:
n_tr = int(0.7 * len(plants))
train_p, test_p = plants.iloc[:n_tr], plants.iloc[n_tr:]
global_mean = train_p["cf"].mean()

means = pd.Series(dtype=float)                # TODO
test_encoded = pd.Series(global_mean, index=test_p.index)    # TODO
n_unseen = np.nan                             # TODO

print("global training mean cf:", round(global_mean, 3))
print("municipalities with an encoding:", len(means))
print("test plants in unseen municipalities:", n_unseen)
print("correlation of the encoded value with the true cf in the test set:", round(test_encoded.corr(test_p["cf"]), 3))

## Summary

* Clean text categories (strip, lowercase, map synonyms) *before* counting levels.
* Check things that must physically agree (capacity × hours ≥ energy) to find unit errors.
* Engineered features change the shape, not the information; test several settings (base temperature) on the training data.
* Pick the rescaling from the shape of the variable, and **fit on training data only**.
* One-hot for few levels, group rare ones first, target-encode very many levels using training data only.